In [ ]:
import os, sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path("..").resolve()))
from config import RAW_DATA_DIR

# all_positions_path = os.path.join(DATA_DIR, "All_Positions20260303.csv")
plf_positions_path = os.path.join(RAW_DATA_DIR, "PLF_Positions20260309.csv")

In [ ]:
# df0 = pd.read_csv(all_positions_path, header=0)
df0 = pd.read_csv(plf_positions_path, header=0)

missing_count = df0['SUPPLEMENTAL_ID_1'].isna().sum()
if missing_count > 0:
    print(f"⚠️ WARNING: {missing_count} rows are missing a SUPPLEMENTAL_ID_1.")
    # Option: Export the broken rows to a desktop CSV to inspect them
    df0[df0['SUPPLEMENTAL_ID_1'].isna()].to_csv("missing_ids_debug.csv")
    
#Strip away the prime brokerage footer on FundID'
df0['FundID'] = df0['FundID'].str.split('-').str[0].str.strip()

#Create a composite primary key that includes the supplemental ID, 
# which is necessary to distinguish between equity and derivates of a fund'
composite_key = [df0['FundID'], df0['SUPPLEMENTAL_ID_1']]

#Instruct pandas to take the first value for all columsn except 'Basket Shares', which we want to sum
agg_dict = {col:'first' for col in df0.columns}

agg_dict['Basket Shares'] = 'sum' 

# We group by our stripped key, apply the logic, and reset the index
df_final = df0.groupby(composite_key).agg(agg_dict).reset_index(drop=True)
df_final.to_csv("PLF_Positions20260309.csv", index=False)